# ONIA — 2.2 Regresia logistică

Notebook de lucru pentru clasificarea binară, log-loss, gradient descent și scikit-learn. Exemplele pot fi executate secvențial. Simulatoarele și explicațiile extinse se găsesc în suportul HTML al lecției.

## 1. Scorul liniar, sigmoidul și predicția

In [ ]:
import math
import numpy as np

def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    e = math.exp(z)
    return e / (1 + e)

for x in [-3, 0, 3]:
    print(x, round(sigmoid(x), 4))

In [ ]:
x = 4
w, b = 1.0, -3.0
z = w * x + b
p = sigmoid(z)
print('Scor:', z, 'Probabilitate:', round(p, 4), 'Clasă:', int(p >= 0.5))

## 2. Log-loss pentru o observație și pentru un set de date

In [ ]:
def log_loss_observatie(y, p):
    eps = 1e-12
    p = min(max(p, eps), 1-eps)
    return -(y * math.log(p) + (1-y) * math.log(1-p))

print(log_loss_observatie(1, 0.9))
print(log_loss_observatie(1, 0.1))

## 3. Implementare completă de la zero

Gradientul unei observații în raport cu scorul este `p-y`. Pentru fiecare epocă adunăm gradienții pe toate observațiile și actualizăm simultan parametrii.

In [ ]:
def antreneaza(x, y, rata=0.12, epoci=1800):
    w = b = 0.0
    istoric = []
    for _ in range(epoci):
        gw = gb = 0.0
        pierdere = 0.0
        for xi, yi in zip(x, y):
            z = w * xi + b
            p = sigmoid(z)
            gw += (p - yi) * xi
            gb += (p - yi)
            pierdere += math.log1p(math.exp(-abs(z))) + max(z,0) - yi*z
        n = len(x)
        w -= rata * gw/n
        b -= rata * gb/n
        istoric.append(pierdere / n)
    return w, b, istoric

x = [-3,-2.5,-2,-1,0,1,2,2.5,3]
y = [0,0,0,0,1,0,1,1,1]
w,b,istoric = antreneaza(x,y)
print('w=',round(w,3),'b=',round(b,3),'loss=',round(istoric[-1],4))

In [ ]:
import matplotlib.pyplot as plt
xs = np.linspace(-4,4,250)
plt.figure(figsize=(7,3.5))
plt.scatter(x,y,c=y,cmap='coolwarm',edgecolor='black')
plt.plot(xs, [sigmoid(w*xx+b) for xx in xs], color='navy')
plt.axhline(0.5,color='gray',linestyle='--')
plt.xlabel('x'); plt.ylabel('p(y=1|x)'); plt.title('Regresia logistică învățată'); plt.show()

In [ ]:
plt.figure(figsize=(7,3))
plt.plot(istoric)
plt.xlabel('Epoca'); plt.ylabel('Log-loss mediu'); plt.title('Convergența gradient descent'); plt.show()

## 4. Model logistic cu mai multe caracteristici în NumPy

In [ ]:
X = np.array([[0,1],[0,2],[1,2],[2,1],[2,2],[3,2],[3,3],[4,3]],dtype=float)
y2 = np.array([0,0,0,0,1,1,1,1],dtype=float)
w2 = np.zeros(X.shape[1]); b2 = 0.0
for _ in range(1200):
    z = X @ w2 + b2
    p = 1 / (1 + np.exp(-np.clip(z,-50,50)))
    e = p - y2
    gw = X.T @ e / len(y2)
    gb = e.mean()
    w2 -= .15*gw
    b2 -= .15*gb
print('Coeficienți:', w2, 'Intercept:', b2)

## 5. Prima utilizare scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(C=1.0, max_iter=1000)
model.fit(X,y2.astype(int))
print('Clase:', model.classes_)
print('Coeficienți:', model.coef_)
print('Probabilități:', model.predict_proba([[1,2],[3,2]]))

## 6. Setul Iris, Pipeline și metrici (fără scurgere de date)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,log_loss
Xi,yi=load_iris(return_X_y=True)
Xi,yi=Xi[:100],yi[:100]
Xtr,Xte,ytr,yte=train_test_split(Xi,yi,test_size=.25,stratify=yi,random_state=42)
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
model.fit(Xtr,ytr)
yp=model.predict(Xte); pp=model.predict_proba(Xte)[:,1]
print('Accuracy',accuracy_score(yte,yp))
print('Precision',precision_score(yte,yp))
print('Recall',recall_score(yte,yp))
print('F1',f1_score(yte,yp))
print('Log-loss',log_loss(yte,pp))
print(confusion_matrix(yte,yp))

## 7. Alegerea pragului pe un set de validare

In [ ]:
for prag in [0.3,0.5,0.7]:
    pred=(pp>=prag).astype(int)
    print('Prag',prag,'Precision',precision_score(yte,pred,zero_division=0),
          'Recall',recall_score(yte,pred,zero_division=0))

## 8. Clasificare neliniară prin caracteristici polinomiale

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
rng=np.random.default_rng(2)
Xc=rng.uniform(-2,2,size=(400,2))
yc=(Xc[:,0]**2+Xc[:,1]**2>2).astype(int)
Xtr,Xte,ytr,yte=train_test_split(Xc,yc,stratify=yc,random_state=42)
linear=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
polinom=make_pipeline(PolynomialFeatures(2,include_bias=False),StandardScaler(),LogisticRegression(max_iter=1000))
for m in [linear,polinom]:
    m.fit(Xtr,ytr)
    print(round(accuracy_score(yte,m.predict(Xte)),3))

## 9. Exerciții de continuare

1. Alege un alt prag și compară precision/recall.
2. Compară `C=0.01`, `C=1` și `C=100` pe datele neliniare.
3. Creează un set cu 95% clasa 0; verifică de ce accuracy poate induce în eroare.
4. Desenează distribuția probabilităților pe clase.